#### Topic 4: Troubleshooting and Tuning Apache Spark DataFrame API Applications
  - Spark UI: stages, tasks, DAG, shuffle read/write
  - explain() — reading physical & logical plans
  - Caching & Persistence: cache(), persist(), storage levels, unpersist()
  - Partitioning: repartition() vs coalesce(), partition count guidelines
  - Shuffles: what causes them, how to minimize
  - Broadcast Joins: broadcast hint, size threshold config
  - Adaptive Query Execution (AQE): coalesce, skew join, dynamic partition pruning
  - Predicate Pushdown & Column Pruning
  - Key Spark configurations for performance tuning
  - Skew detection and salting pattern

In [0]:
"""
Databricks Certified Associate Developer
Topic 4: Troubleshooting and Tuning Apache Spark DataFrame API Applications (10%)
  - Spark UI: stages, tasks, DAG, shuffle read/write
  - explain() — reading physical & logical plans
  - Caching & Persistence: cache(), persist(), storage levels, unpersist()
  - Partitioning: repartition() vs coalesce(), partition count guidelines
  - Shuffles: what causes them, how to minimize
  - Broadcast Joins: broadcast hint, size threshold config
  - Adaptive Query Execution (AQE): coalesce, skew join, dynamic partition pruning
  - Predicate Pushdown & Column Pruning
  - Key Spark configurations for performance tuning
  - Skew detection and salting pattern
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, broadcast, spark_partition_id, count, lit, concat, rand, floor,
    explode, array, monotonically_increasing_id
)
from pyspark.storagelevel import StorageLevel

spark = SparkSession.builder \
    .appName("SparkTuning_Troubleshooting") \
    .master("local[*]") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# ---------------------------------------------------------------------------
# Sample data
# ---------------------------------------------------------------------------
orders_data = [
    (1, 101, "2024-01-05", 250.0),
    (2, 102, "2024-01-07", 450.0),
    (3, 101, "2024-01-10", 300.0),
    (4, 103, "2024-01-12", 800.0),
    (5, 102, "2024-01-15", 150.0),
    (6, 104, "2024-01-18", 600.0),
    (7, 101, "2024-01-20", 200.0),
    (8, 105, "2024-01-22", 950.0),
]
orders_df = spark.createDataFrame(
    orders_data, "order_id INT, customer_id INT, order_date STRING, amount DOUBLE"
)

customers_data = [
    (101, "Alice",   "New York"),
    (102, "Bob",     "Chicago"),
    (103, "Carol",   "Dallas"),
    (104, "Dave",    "Boston"),
    (105, "Eve",     "Seattle"),
]
customers_df = spark.createDataFrame(
    customers_data, "customer_id INT, name STRING, city STRING"
)

orders_df.createOrReplaceTempView("orders")
customers_df.createOrReplaceTempView("customers")

# ---------------------------------------------------------------------------
# 1. explain() — READING EXECUTION PLANS
#
#  explain(mode)  modes:
#    "simple"    → physical plan only (default)
#    "extended"  → parsed → analyzed → optimized → physical
#    "codegen"   → generated Java bytecode
#    "cost"      → with statistics (requires ANALYZE TABLE)
#    "formatted" → human-readable, indented physical plan
#
#  KEY plan operators to recognise on the exam:
#    FileScan           → reading from disk
#    Filter             → WHERE clause applied
#    Project            → SELECT (column selection)
#    HashAggregate      → aggregation (2-phase: partial + final)
#    Exchange           → SHUFFLE (= network data movement)
#    BroadcastExchange  → small table sent to all executors
#    SortMergeJoin      → large-large join (after sort + shuffle)
#    BroadcastHashJoin  → large-small join (no shuffle on small side)
# ---------------------------------------------------------------------------
print("=== 1. explain() — Physical Plan ===")
join_query = orders_df.join(customers_df, "customer_id").filter(col("amount") > 200)
join_query.explain()                    # simple (physical only)

print("=== 1b. explain('formatted') ===")
join_query.explain("formatted")

print("=== 1c. explain('extended') — all 4 plan stages ===")
join_query.explain("extended")

# ---------------------------------------------------------------------------
# 2. CACHING & PERSISTENCE
#
#  cache()              → shortcut for persist(StorageLevel.MEMORY_AND_DISK)
#                         (in Spark 3.x, defaults to MEMORY_AND_DISK_DESER)
#  persist(level)       → explicit storage level
#  unpersist()          → free cached memory immediately (best practice)
#  df.is_cached         → True/False (DataFrame API)
#
#  StorageLevel options (know the names & trade-offs):
#    MEMORY_ONLY          → RDD / Python objects in RAM; re-computes on eviction
#    MEMORY_AND_DISK      → spills to disk if RAM full (most common)
#    DISK_ONLY            → always on disk; slowest
#    MEMORY_ONLY_SER      → serialized bytes in RAM; less GC pressure
#    MEMORY_AND_DISK_SER  → serialized, spills to disk
#    OFF_HEAP             → outside JVM heap; requires spark.memory.offHeap.enabled
#
#  WHEN TO CACHE:
#    ✓ DataFrame used multiple times in the same job
#    ✗ DataFrame used only once — caching costs more than it saves
#    ✗ Very large DataFrames that exceed available memory
# ---------------------------------------------------------------------------
print("=== 2. Caching & Persistence ===")

# cache() is lazy — actual caching happens on the first ACTION
orders_df.cache()
print("is_cached after cache():", orders_df.is_cached)

# Trigger the cache with an action
orders_df.count()

# persist() with explicit storage level
customers_df.persist(StorageLevel.MEMORY_AND_DISK)
customers_df.count()

print("cached orders count:", orders_df.count())     # served from cache
print("cached customers count:", customers_df.count())

# Always unpersist to release memory when done
orders_df.unpersist()
customers_df.unpersist()
print("is_cached after unpersist():", orders_df.is_cached)

# ---------------------------------------------------------------------------
# 3. PARTITIONING — repartition() vs coalesce()
#
#  repartition(n)          → FULL SHUFFLE, creates exactly n partitions
#  repartition(n, col)     → shuffle + hash on column (even dist. per key)
#  coalesce(n)             → NO full shuffle, merges partitions on same node
#                            (only decreases partition count)
#
#  KEY RULES:
#    coalesce(n)  → only for REDUCING  partitions (no shuffle = fast)
#    repartition  → for INCREASING or BALANCING partitions (full shuffle)
#    coalesce(1)  → single-file write; avoids shuffle BUT creates skew
#    repartition(1) → same single file but triggers full shuffle first
#
#  PARTITION COUNT GUIDELINE:
#    Target: 2–3x number of CPU cores (spark.default.parallelism)
#    Target partition size: 128 MB – 200 MB
#    spark.sql.shuffle.partitions (default: 200) governs post-shuffle partitions
# ---------------------------------------------------------------------------
print("=== 3. Partitioning ===")

# Check current partition count
large_df = spark.range(0, 100_000).toDF("id")
print("Default partitions:", large_df.rdd.getNumPartitions())

# repartition — full shuffle to exactly 8 partitions
repartitioned = large_df.repartition(8)
print("After repartition(8):", repartitioned.rdd.getNumPartitions())

# coalesce — merge without shuffle (only reduce)
coalesced = repartitioned.coalesce(2)
print("After coalesce(2):", coalesced.rdd.getNumPartitions())

# repartition by column — good for downstream joins on that key
orders_by_customer = orders_df.repartition(4, col("customer_id"))
print("repartition by customer_id:", orders_by_customer.rdd.getNumPartitions())

# Check partition skew (rows per partition)
print("=== 3b. Rows per partition ===")
orders_by_customer \
    .withColumn("partition_id", spark_partition_id()) \
    .groupBy("partition_id") \
    .agg(count("*").alias("row_count")) \
    .orderBy("partition_id") \
    .show()

# ---------------------------------------------------------------------------
# 4. SHUFFLES — the most expensive operation in Spark
#
#  Operations that CAUSE a shuffle (data moves across nodes):
#    - groupBy / agg / reduceByKey
#    - join (SortMergeJoin)
#    - repartition(n)
#    - distinct()
#    - orderBy / sort (global sort)
#    - window functions with PARTITION BY
#
#  Operations that do NOT cause a shuffle:
#    - filter / where
#    - select / withColumn
#    - coalesce (only reduces, no network move)
#    - mapPartitions
#    - broadcast join (no shuffle on large side)
#
#  Key config: spark.sql.shuffle.partitions (default 200)
#    → Too high: many tiny tasks, scheduling overhead
#    → Too low:  large tasks, GC pressure, potential OOM
# ---------------------------------------------------------------------------
print("=== 4. Shuffle Partitions Config ===")

# Show current setting
print("spark.sql.shuffle.partitions:",
      spark.conf.get("spark.sql.shuffle.partitions"))

# Override for this session (tune to cluster size)
spark.conf.set("spark.sql.shuffle.partitions", "8")

# This groupBy triggers a shuffle — Exchange node appears in explain()
print("=== 4b. GroupBy — triggers shuffle ===")
orders_df.groupBy("customer_id").agg(count("*").alias("order_cnt")).explain()

# ---------------------------------------------------------------------------
# 5. BROADCAST JOINS
#
#  Default threshold: spark.sql.autoBroadcastJoinThreshold = 10 MB
#  → Spark automatically broadcasts tables below this size.
#
#  Manual broadcast: use broadcast() hint or SQL /*+ BROADCAST(t) */
#
#  BroadcastHashJoin vs SortMergeJoin:
#    BroadcastHashJoin → small table sent to ALL executors; NO shuffle on large side
#    SortMergeJoin     → both sides sorted and shuffled; used for large-large joins
#
#  When to broadcast manually:
#    ✓ Small lookup/dimension tables (cities, categories, etc.)
#    ✗ Both tables are large (waste of driver/executor memory)
# ---------------------------------------------------------------------------
print("=== 5. Broadcast Join ===")

# Force a broadcast of the small customers table
broadcast_join = orders_df.join(
    broadcast(customers_df),   # <-- broadcast hint
    "customer_id"
)
broadcast_join.explain()   # should show BroadcastHashJoin, NOT SortMergeJoin
broadcast_join.show()

# SQL equivalent broadcast hint
print("=== 5b. SQL BROADCAST hint ===")
spark.sql("""
    SELECT /*+ BROADCAST(c) */
           o.order_id, o.amount, c.name, c.city
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
""").explain()

# Disable auto-broadcast (force SortMergeJoin for demo)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
print("=== 5c. SortMergeJoin (auto-broadcast disabled) ===")
orders_df.join(customers_df, "customer_id").explain()

# Re-enable
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

# ---------------------------------------------------------------------------
# 6. PREDICATE PUSHDOWN & COLUMN PRUNING
#
#  Predicate Pushdown:
#    Catalyst optimizer moves WHERE filters as close to the data source
#    as possible — ideally into the file scan itself.
#    → Reduces data read from disk (Parquet, Delta, JDBC, etc.)
#    → Visible in explain() as "PushedFilters" inside FileScan
#
#  Column Pruning:
#    Only columns referenced in the query are read from disk.
#    → Parquet/Delta store data column-by-column → huge savings.
#
#  BEST PRACTICES:
#    ✓ Filter early (as close to source as possible)
#    ✓ Select only needed columns (avoid SELECT *)
#    ✓ Use Parquet/Delta (columnar = supports both optimizations)
#    ✗ UDFs block predicate pushdown (Catalyst cannot inspect them)
# ---------------------------------------------------------------------------
print("=== 6. Predicate Pushdown & Column Pruning ===")

# Both filter and column selection are visible in the optimized plan
orders_df \
    .filter(col("amount") > 300) \
    .select("order_id", "customer_id", "amount") \
    .explain("extended")

# ---------------------------------------------------------------------------
# 7. ADAPTIVE QUERY EXECUTION (AQE) — Spark 3.0+
#
#  AQE re-optimizes the query DURING execution using runtime statistics.
#  Three main features:
#
#  a) Coalescing post-shuffle partitions
#     spark.sql.adaptive.coalescePartitions.enabled (default: true in 3.2+)
#     → Merges small shuffle partitions at runtime → fewer, larger tasks
#     → Controlled by: spark.sql.adaptive.advisoryPartitionSizeInBytes (default 64MB)
#
#  b) Converting SortMergeJoin → BroadcastHashJoin at runtime
#     spark.sql.adaptive.localShuffleReader.enabled
#     → If one side shrinks below autoBroadcastJoinThreshold after a shuffle,
#       AQE broadcasts it dynamically.
#
#  c) Skew Join Optimization
#     spark.sql.adaptive.skewJoin.enabled (default: true)
#     → Splits skewed partitions into smaller sub-partitions automatically
#     → spark.sql.adaptive.skewJoin.skewedPartitionFactor (default: 5)
#     → spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes (default: 256MB)
# ---------------------------------------------------------------------------
print("=== 7. AQE — current settings ===")
aqe_configs = [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",
]
for c in aqe_configs:
    try:
        print(f"  {c} = {spark.conf.get(c)}")
    except Exception:
        print(f"  {c} = (not set / default)")

# ---------------------------------------------------------------------------
# 8. DATA SKEW — detection and salting technique
#
#  Skew: one partition contains significantly more data than others.
#  Symptoms: one task takes 10x longer than others (visible in Spark UI).
#
#  Causes:
#    - Highly imbalanced key distribution (e.g., NULL, "Unknown" dominate)
#    - repartition by a low-cardinality column
#
#  Solutions:
#    1. AQE skew join (automatic in Spark 3.0+)
#    2. Salting — artificially add random suffix to skewed keys
#       → Distributes one logical key across multiple partitions
#       → Requires exploding the small-side key to match all suffixes
#    3. Filter skewed keys and join separately, then union results
# ---------------------------------------------------------------------------
print("=== 8. Skew Detection ===")

# Simulate skewed data: customer 101 has 80% of orders
skewed_data = [(101, i, 10.0) for i in range(80)] + \
              [(102, i + 80, 20.0) for i in range(10)] + \
              [(103, i + 90, 30.0) for i in range(10)]
skewed_orders = spark.createDataFrame(
    skewed_data, "customer_id INT, order_id INT, amount DOUBLE"
).repartition(4, col("customer_id"))

print("Rows per partition (skewed):")
skewed_orders \
    .withColumn("pid", spark_partition_id()) \
    .groupBy("pid") \
    .agg(count("*").alias("rows")) \
    .orderBy("pid") \
    .show()

# --- Salting technique ---
SALT_BUCKETS = 4

# Step 1: Add random salt to the large (skewed) side
salted_orders = skewed_orders.withColumn(
    "salt",
    (floor(rand() * SALT_BUCKETS)).cast("int")
).withColumn(
    "salted_customer_id",
    concat(col("customer_id").cast("string"), lit("_"), col("salt").cast("string"))
)

# Step 2: Explode the small side to match every salt bucket
small_side = customers_df.withColumn(
    "salt_array",
    array([lit(i) for i in range(SALT_BUCKETS)])
).withColumn(
    "salt", explode(col("salt_array"))
).withColumn(
    "salted_customer_id",
    concat(col("customer_id").cast("string"), lit("_"), col("salt").cast("string"))
).drop("salt_array", "salt", "customer_id")

# Step 3: Join on salted key
salted_result = salted_orders.join(
    broadcast(small_side),
    "salted_customer_id"
).drop("salted_customer_id", "salt")

print("=== 8b. Salted join result (first 5 rows) ===")
salted_result.select("order_id", "customer_id", "name", "amount").show(5)

# ---------------------------------------------------------------------------
# 9. KEY SPARK CONFIGURATIONS FOR PERFORMANCE
# ---------------------------------------------------------------------------
print("=== 9. Key Configurations ===")

tuning_configs = {
    "spark.sql.shuffle.partitions":
        "Post-shuffle partition count (default 200; tune to 2-3x cores)",
    "spark.sql.autoBroadcastJoinThreshold":
        "Auto-broadcast threshold in bytes (default 10MB; -1 = disable)",
    "spark.sql.adaptive.enabled":
        "Enable AQE (default true in Spark 3.2+)",
    "spark.sql.adaptive.coalescePartitions.enabled":
        "AQE merges small post-shuffle partitions",
    "spark.sql.adaptive.skewJoin.enabled":
        "AQE splits skewed join partitions",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes":
        "Target size for coalesced partitions (default 64MB)",
    "spark.executor.memory":
        "Total executor memory (e.g. 4g); split: 60% execution / 40% storage",
    "spark.executor.cores":
        "Cores per executor (2-5 recommended; more = more parallelism but more GC)",
    "spark.memory.fraction":
        "Fraction of executor memory for Spark (default 0.6)",
    "spark.memory.storageFraction":
        "Storage fraction within spark.memory.fraction (default 0.5)",
    "spark.default.parallelism":
        "Default parallelism for RDD operations (set to 2-3x total cores)",
    "spark.sql.files.maxPartitionBytes":
        "Max bytes per partition when reading files (default 128MB)",
    "spark.serializer":
        "Serializer: org.apache.spark.serializer.KryoSerializer (faster than Java)",
}

for key, desc in tuning_configs.items():
    print(f"  {key}:\n    → {desc}")

# ---------------------------------------------------------------------------
# 10. COMMON TROUBLESHOOTING PATTERNS
# ---------------------------------------------------------------------------
print("=== 10a. Avoid wide transformations when narrow is sufficient ===")

# BAD: groupBy triggers shuffle even when you just need total count
orders_df.groupBy().count().explain()   # Exchange present

# BETTER for simple aggregation on cached data: use count action directly
print("Row count (no shuffle plan):", orders_df.count())

print("=== 10b. Pushdown filter before join ===")

# Without early filter — join processes all rows first
plan_late_filter = orders_df \
    .join(customers_df, "customer_id") \
    .filter(col("amount") > 400)

# With early filter — reduces rows before the expensive join
plan_early_filter = orders_df \
    .filter(col("amount") > 400) \
    .join(customers_df, "customer_id")

print("Late filter plan:")
plan_late_filter.explain()
print("Early filter plan (fewer rows into join):")
plan_early_filter.explain()

print("=== 10c. Detect and drop duplicates efficiently ===")
# dropDuplicates() triggers a shuffle; use it only on necessary columns
orders_df.dropDuplicates(["customer_id"]).explain()

print("=== 10d. Avoid collect() on large DataFrames ===")
# collect() brings ALL data to driver — causes OOM for large data
# Use show(), take(n), or write to storage instead
orders_df.show(3)           # safe — only fetches top n rows
orders_df.take(3)           # returns list of Row objects to driver (small n only)

# ---------------------------------------------------------------------------
# KEY EXAM FACTS — Troubleshooting and Tuning
# ---------------------------------------------------------------------------
"""
EXAM QUICK REFERENCE:
─────────────────────────────────────────────────────────────────────────
CACHING:
  cache()              = persist(MEMORY_AND_DISK)  ← lazy (needs action)
  StorageLevel options: MEMORY_ONLY, MEMORY_AND_DISK, DISK_ONLY, *_SER, OFF_HEAP
  Always unpersist() when done; df.is_cached to check

PARTITIONING:
  repartition(n)       → full shuffle, can increase or decrease
  coalesce(n)          → no shuffle, can only DECREASE partitions
  repartition(n, col)  → shuffle + hash partition on column

SHUFFLE-CAUSING OPERATIONS:
  groupBy, join (SMJ), repartition, distinct, orderBy, window PARTITION BY

BROADCAST JOIN:
  broadcast(df)  or  SELECT /*+ BROADCAST(t) */ ...
  Threshold: spark.sql.autoBroadcastJoinThreshold (default 10MB)
  Plan shows BroadcastHashJoin (not SortMergeJoin)

AQE (Adaptive Query Execution) — Spark 3.0+:
  spark.sql.adaptive.enabled = true
  ① Coalesces small post-shuffle partitions at runtime
  ② Converts SortMergeJoin → BroadcastHashJoin dynamically
  ③ Splits skewed join partitions automatically

EXPLAIN PLAN NODES TO KNOW:
  Exchange              → shuffle (expensive, network I/O)
  BroadcastExchange     → small table broadcast
  BroadcastHashJoin     → efficient join (no shuffle on large side)
  SortMergeJoin         → large-large join (both sides shuffled + sorted)
  HashAggregate         → 2-phase aggregation (partial then final)
  Filter / Project      → WHERE / SELECT (should appear early = pushdown)

PREDICATE PUSHDOWN & COLUMN PRUNING:
  → Parquet/Delta supports both → always prefer columnar formats
  → UDFs BLOCK predicate pushdown
  → Visible as "PushedFilters" in FileScan node of explain() output

SKEW:
  Symptom: one stage task takes much longer than others (Spark UI)
  Fix 1: AQE (spark.sql.adaptive.skewJoin.enabled = true)
  Fix 2: Salting — add random suffix to skewed key; explode small side

MEMORY:
  spark.executor.memory       → total executor memory
  spark.memory.fraction       → fraction usable by Spark (default 0.6)
  spark.memory.storageFraction→ cache share within that fraction (default 0.5)
─────────────────────────────────────────────────────────────────────────
"""

spark.stop()
